For faster downloads and for better GPU, use colab extension on VS Code to connect to a remote Colab Instance

# Checking the Non domain-adapted model

ModernBERT-base is used for its benchmarks and size.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments, default_data_collator
from datasets import Dataset
import collections
import numpy as np
import math

SEED = 67
CHUNK_SIZE = 128

In [2]:

model_use = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_use)
model = AutoModelForMaskedLM.from_pretrained(model_use)

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

Attempt on an arbitrary text on a news body

In [3]:
test = "The Department of Health (DOH) on Thursday night, July 22, confirmed the local [MASK] of the highly transmissible COVID-19 Delta variant in the Philippines."

In [4]:
inputs = tokenizer(test, return_tensors='pt')
outputs = model(**inputs)
logits = outputs.logits

In [5]:
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = logits[0, mask_token_index, :]
top_10_tokens = torch.topk(mask_token_logits, 10, dim = 1).indices[0].tolist()
for token in top_10_tokens:
    print(f"{tokenizer.decode([token])}, {test.replace(tokenizer.mask_token, tokenizer.decode(token).strip())}")

 presence, The Department of Health (DOH) on Thursday night, July 22, confirmed the local presence of the highly transmissible COVID-19 Delta variant in the Philippines.
 emergence, The Department of Health (DOH) on Thursday night, July 22, confirmed the local emergence of the highly transmissible COVID-19 Delta variant in the Philippines.
 outbreak, The Department of Health (DOH) on Thursday night, July 22, confirmed the local outbreak of the highly transmissible COVID-19 Delta variant in the Philippines.
 appearance, The Department of Health (DOH) on Thursday night, July 22, confirmed the local appearance of the highly transmissible COVID-19 Delta variant in the Philippines.
 arrival, The Department of Health (DOH) on Thursday night, July 22, confirmed the local arrival of the highly transmissible COVID-19 Delta variant in the Philippines.
 detection, The Department of Health (DOH) on Thursday night, July 22, confirmed the local detection of the highly transmissible COVID-19 Delta va

## Preprocessing the Dataset

In [6]:
import json
import pandas as pd

with open('../scraping/rappler_articles.json') as rappler_f:
    rappler_json = json.load(rappler_f)

rappler_df = pd.DataFrame(rappler_json['scraped'])

rappler_df

,title,body
0,Basagan ng Trip with Leloy Claudio: Heroism in...,"MANILA, Philippines – Who is your hero? Can y..."
1,DOH confirms local transmission of COVID-19 De...,The Department of Health (DOH) on Thursday nig...
2,Robin Padilla urges Filipinos to patronize loc...,"MANILA, Philippines – Robin Padilla is known ..."
3,"Pacquiao, Cayetano: Matobato lies ‘damaged’ PH...","MANILA, Philippines – Staunch allies of Presi..."
4,"Bucks erase fourth-quarter deficit, stun Celtics",Bobby Portis turned a missed Giannis Antetokou...
...,...,...
995,A prayer for climate,"Today, September 1, is marked by the Catholic ..."
996,Tim Cone wishes Justin Brownlee well ahead of ...,"MANILA, Philippines – Barangay Ginebra residen..."
997,Olympic hero Farah slams ‘ignorance and prejud...,"LONDON, England – British athletics legend Mo..."
998,UN fears stigmatization of rescued Boko Haram ...,"GENEVA, Switzerland – The United Nations voice..."


In [7]:
rappler_df.head()

,title,body
0,Basagan ng Trip with Leloy Claudio: Heroism in...,"MANILA, Philippines – Who is your hero? Can y..."
1,DOH confirms local transmission of COVID-19 De...,The Department of Health (DOH) on Thursday nig...
2,Robin Padilla urges Filipinos to patronize loc...,"MANILA, Philippines – Robin Padilla is known ..."
3,"Pacquiao, Cayetano: Matobato lies ‘damaged’ PH...","MANILA, Philippines – Staunch allies of Presi..."
4,"Bucks erase fourth-quarter deficit, stun Celtics",Bobby Portis turned a missed Giannis Antetokou...


In [8]:
def tokenize_func(data):
    tokenized = tokenizer(data["body"])
    if tokenizer.is_fast:
        tokenized["word_ids"] = [tokenized.word_ids(i) for i in range(len(tokenized["input_ids"]))]
    return tokenized


rappler_ds = Dataset.from_pandas(rappler_df)
rappler_ds = rappler_ds.train_test_split(seed=SEED)

tokenized_rappler = rappler_ds.map(
    tokenize_func, batched=True, remove_columns=['title', 'body']
)
tokenized_rappler


Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids'],
        num_rows: 750
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids'],
        num_rows: 250
    })
})

Concatenate all articles and split each article into texts of length chunk size.

In [9]:
def concatenate_chunk(data):
    concatenated = {key: sum(data[key], []) for key in data}
    total_length = len(concatenated[list(data)[0]])
    # remove last when smaller than chunk size
    total_length = (total_length // CHUNK_SIZE) * CHUNK_SIZE
    # split
    splitted = {
        key: [row[i:i + CHUNK_SIZE] for i in range(0, total_length, CHUNK_SIZE)]
        for key, row in concatenated.items()
    }

    splitted["labels"] = splitted["input_ids"].copy()

    return splitted

In [10]:
rappler_chunked = tokenized_rappler.map(concatenate_chunk, batched=True)
rappler_chunked

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 4101
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 1225
    })
})

In [11]:
tokenizer.decode(rappler_chunked["train"][0]["input_ids"])

'[CLS] PRAIA, Cape Verde – African countries agreed Saturday, February 14, to strengthen their meteorological services to reduce the impact of extreme weather events at a meeting of ministers in Cape Verde. In a declaration adopted at the end of the 5-day African Ministerial Conference on Meteorology (AMCOMET), delegates recognized that “investments in weather and climate services help save lives and property, minimize economic losses and preserve the environment.” Part of the discussion focused on recent natural disasters on the continent, such as deadly floods in January in Malawi and Mozambique. The participants adopted a budget of around one million euros'

## Fine-tuning

In [12]:
# perform whole word masking
